In [1]:
import json
from pathlib import Path

import torch
import numpy as np

SEED = 42
MAX_LENGTH = 64
MODEL_NAME = "indobenchmark/indobert-base-p1"

In [2]:
label_map = json.loads(Path("../data/ner/label_map.json").read_text())
label2id = {k: int(v) for k, v in label_map.items()}
id2label = {v: k for k, v in label2id.items()}
LABEL_LIST = [id2label[i] for i in range(len(id2label))]
print(LABEL_LIST)

['O', 'B-FOOD_ITEM', 'I-FOOD_ITEM', 'B-DRINK_ITEM', 'I-DRINK_ITEM', 'B-QUANTITY', 'I-QUANTITY', 'B-MODIFIER', 'I-MODIFIER']


## Step 1 — Load splits

In [3]:
from datasets import Dataset

def load_split(jsonl_path: str) -> Dataset:
    records = []
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            records.append({
                "input_ids": rec["input_ids"],
                "attention_mask": rec["attention_mask"],
                "labels": rec["labels"],
            })
    return Dataset.from_list(records)


train_dataset = load_split("../data/ner/train.jsonl")
val_dataset   = load_split("../data/ner/val.jsonl")
test_dataset  = load_split("../data/ner/test.jsonl")

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1267
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 271
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 272
})


## Step 2 — Class imbalance

Entity tags skew toward `O` and toward common tags (FOOD_ITEM, QUANTITY) vs the rarer `MODIFIER`/`DRINK_ITEM`. Inverse-frequency class weights computed from the **train split only**, same convention as `notebooks/3.2`'s Step 4.

In [4]:
from collections import Counter

def compute_label_counts(jsonl_path: str) -> Counter:
    """Count label ids across a split, excluding -100 (ignored/padding)."""
    counts = Counter()
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            for lbl_id in rec["labels"]:
                if lbl_id != -100:
                    counts[lbl_id] += 1
    return counts


train_label_counts = compute_label_counts("../data/ner/train.jsonl")
for lbl_id in range(len(LABEL_LIST)):
    print(f"  {id2label[lbl_id]:14s}: {train_label_counts[lbl_id]:6d}")

  O             :  24214
  B-FOOD_ITEM   :   2607
  I-FOOD_ITEM   :   2713
  B-DRINK_ITEM  :    396
  I-DRINK_ITEM  :    397
  B-QUANTITY    :    437
  I-QUANTITY    :      0
  B-MODIFIER    :    784
  I-MODIFIER    :    851


In [5]:
total_tokens = sum(train_label_counts.values())
n_classes = len(LABEL_LIST)

class_weights = torch.tensor(
    [total_tokens / (n_classes * max(train_label_counts[i], 1)) for i in range(n_classes)],
    dtype=torch.float,
)

for lbl_id, weight in enumerate(class_weights.tolist()):
    print(f"  {id2label[lbl_id]:14s}: weight={weight:.3f}")

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)


  O             : weight=0.149
  B-FOOD_ITEM   : weight=1.381
  I-FOOD_ITEM   : weight=1.327
  B-DRINK_ITEM  : weight=9.091
  I-DRINK_ITEM  : weight=9.068
  B-QUANTITY    : weight=8.238
  I-QUANTITY    : weight=3599.889
  B-MODIFIER    : weight=4.592
  I-MODIFIER    : weight=4.230


### Sanity check

In [6]:
assert class_weights.argmin().item() == label2id["O"], "O should get the lowest weight"
assert loss_fn.ignore_index == -100
print("Step 2 sanity checks passed.")


Step 2 sanity checks passed.


## Step 3 — Model architecture

`BertCrfForTokenClassification` (`../src/ordo_ai/models/ner_crf.py`): full fine-tune of `indobenchmark/indobert-base-p1` encoder + linear emissions + a CRF decoding layer. The CRF's learned transition matrix forbids invalid BIO transitions (e.g. `I-X` with no preceding `B-X`) that plain per-token argmax cannot prevent.

In [7]:
import sys
sys.path.insert(0, "../src")
from ordo_ai.models.ner_crf import BertCrfForTokenClassification

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

model = BertCrfForTokenClassification.from_pretrained_bert(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"device          : {device}")
print(f"total params    : {n_params:,}")
print(f"trainable params: {n_trainable:,}  (full fine-tune — should equal total)")


[transformers] You passed `num_labels=9` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

device          : mps
total params    : 124,448,364
trainable params: 124,448,364  (full fine-tune — should equal total)


### Sanity check: forward pass on a real batch

In [8]:
sample_batch = train_dataset[:4]
batch_input_ids = torch.tensor(sample_batch["input_ids"], device=device)
batch_attention_mask = torch.tensor(sample_batch["attention_mask"], device=device)
batch_labels = torch.tensor(sample_batch["labels"], device=device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask, labels=batch_labels)

logits = outputs["logits"]
print(f"logits shape: {tuple(logits.shape)}  (expect (4, {MAX_LENGTH}, {len(LABEL_LIST)}))")
assert logits.shape == (4, MAX_LENGTH, len(LABEL_LIST))

print(f"sample CRF negative log-likelihood loss: {outputs['loss'].item():.4f}")

model.train()
print("Step 3 sanity checks passed.")


logits shape: (4, 64, 9)  (expect (4, 64, 9))
sample CRF negative log-likelihood loss: 75.0195
Step 3 sanity checks passed.


## Step 4 — Training

`CrfTrainer` overrides `compute_loss` to use the CRF negative-log-likelihood loss (Step 3) instead of `Trainer`'s per-token cross-entropy default, and overrides `prediction_step` to Viterbi-decode through the CRF layer for eval/predict instead of per-token argmax.

In [9]:
import evaluate

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    """
    Convert per-token Viterbi-decoded predictions/labels back into BIO tag
    sequences (dropping -100 positions) and score with seqeval's
    entity-level precision/recall/F1.
    """
    preds, labels = eval_pred

    true_seqs, pred_seqs = [], []
    for pred_row, label_row in zip(preds, labels):
        true_seq = [id2label[l] for l in label_row if l != -100]
        pred_seq = [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
        true_seqs.append(true_seq)
        pred_seqs.append(pred_seq)

    results = seqeval_metric.compute(predictions=pred_seqs, references=true_seqs)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [10]:
from ordo_ai.tracking import init_experiment

init_experiment("menu-ner")

2026/07/02 15:17:27 INFO mlflow.tracking.fluent: Experiment with name 'menu-ner' does not exist. Creating a new experiment.


In [11]:
from transformers import Trainer, TrainingArguments

class CrfTrainer(Trainer):
    """Trainer using the model's built-in CRF negative-log-likelihood loss
    (Step 3) instead of `Trainer`'s default per-token cross-entropy. Eval/predict
    steps Viterbi-decode through the CRF layer instead of taking per-token argmax,
    since only Viterbi enforces valid BIO transitions at inference time too."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        labels = inputs.get("labels")

        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs["loss"].detach() if labels is not None else None
            decoded = model.decode(inputs["input_ids"], inputs["attention_mask"])

        seq_len = inputs["input_ids"].shape[1]
        preds = torch.zeros((len(decoded), seq_len), dtype=torch.long)
        for i, tag_seq in enumerate(decoded):
            preds[i, : len(tag_seq)] = torch.tensor(tag_seq, dtype=torch.long)

        if prediction_loss_only:
            return (loss, None, None)
        return (loss, preds, labels)


training_args = TrainingArguments(
    output_dir="../models/indobert-ner-bio",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="epoch",
    report_to=["mlflow"],
    seed=SEED,
)

trainer = CrfTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
import mlflow

run = mlflow.start_run()
train_result = trainer.train()
train_result.metrics


/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,14.694272,0.959996,0.949353,0.950378,0.949865,0.989660
2,0.676652,0.481880,0.974359,0.983819,0.979066,0.996122
3,0.178234,0.382327,0.984946,0.988134,0.986537,0.997559
4,0.080655,0.420421,0.984930,0.987055,0.985991,0.997559
5,0.020650,0.410622,0.983871,0.987055,0.985460,0.997559


/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752:

{'train_runtime': 239.4369,
 'train_samples_per_second': 26.458,
 'train_steps_per_second': 1.671,
 'total_flos': 208364402695680.0,
 'train_loss': 3.1300925326347353,
 'epoch': 5.0}

### Save the fine-tuned model

In [13]:
from transformers import AutoTokenizer

final_model_path = "../models/indobert-ner-bio-final"
trainer.save_model(final_model_path)
AutoTokenizer.from_pretrained(MODEL_NAME).save_pretrained(final_model_path)
print(f"Saved fine-tuned model -> {final_model_path}")

mlflow.log_artifact(final_model_path, artifact_path="model")
mlflow.end_run()

Saved fine-tuned model -> ../models/indobert-ner-bio-final
🏃 View run orderly-hen-764 at: http://localhost:5001/#/experiments/2/runs/47505db6970c462b887fb4c7254a7c7d
🧪 View experiment at: http://localhost:5001/#/experiments/2


## Step 5 — Evaluation

Run on the held-out **test** split — not seen during training or `load_best_model_at_end` checkpoint selection — and report per-entity-type precision/recall/F1, not just overall, since rarer tags (MODIFIER, DRINK_ITEM) are where this model is most likely to be weak.

In [14]:
test_predictions = trainer.predict(test_dataset)
test_preds, test_label_ids = test_predictions.predictions, test_predictions.label_ids

print("overall test metrics:")
for k, v in test_predictions.metrics.items():
    print(f"  {k:25s}: {v}")


/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


overall test metrics:
  test_loss                : 0.49260953068733215
  test_precision           : 0.98068669527897
  test_recall              : 0.9902491874322861
  test_f1                  : 0.98544474393531
  test_accuracy            : 0.9966911235793411
  test_runtime             : 4.6354
  test_samples_per_second  : 58.679
  test_steps_per_second    : 3.667


In [15]:
test_true_seqs, test_pred_seqs = [], []
for pred_row, label_row in zip(test_preds, test_label_ids):
    true_seq = [id2label[l] for l in label_row if l != -100]
    pred_seq = [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
    test_true_seqs.append(true_seq)
    test_pred_seqs.append(pred_seq)

per_entity_report = seqeval_metric.compute(
    predictions=test_pred_seqs, references=test_true_seqs
)

print("Per-entity-type breakdown (test set):")
for entity_type, scores in per_entity_report.items():
    if entity_type.startswith("overall"):
        continue
    print(
        f"  {entity_type:10s}  precision={scores['precision']:.3f}  "
        f"recall={scores['recall']:.3f}  f1={scores['f1']:.3f}  "
        f"n={scores['number']}"
    )
print()
print(f"  overall precision={per_entity_report['overall_precision']:.3f}  "
      f"recall={per_entity_report['overall_recall']:.3f}  "
      f"f1={per_entity_report['overall_f1']:.3f}  "
      f"accuracy={per_entity_report['overall_accuracy']:.3f}")


Per-entity-type breakdown (test set):
  DRINK_ITEM  precision=1.000  recall=1.000  f1=1.000  n=86
  FOOD_ITEM   precision=0.988  recall=0.996  f1=0.992  n=561
  MODIFIER    precision=0.958  recall=0.958  f1=0.958  n=167
  QUANTITY    precision=0.965  recall=1.000  f1=0.982  n=109

  overall precision=0.981  recall=0.990  f1=0.985  accuracy=0.997


### Misclassified examples

In [16]:
source_by_id = {}
with open("../data/normalized/augmented_ordering_dataset_normalized.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        source_by_id[rec["id"]] = rec["text_normalized"].split()

test_ids = []
with open("../data/ner/test.jsonl", encoding="utf-8") as f:
    for line in f:
        test_ids.append(json.loads(line)["id"])

n_mismatch = 0
for rec_id, true_seq, pred_seq in zip(test_ids, test_true_seqs, test_pred_seqs):
    if true_seq != pred_seq:
        n_mismatch += 1
print(f"Mismatched records: {n_mismatch} / {len(test_ids)}\n")

shown = 0
for rec_id, true_seq, pred_seq in zip(test_ids, test_true_seqs, test_pred_seqs):
    if true_seq == pred_seq or shown >= 10:
        continue
    tokens = source_by_id[rec_id]
    print(f"id={rec_id}")
    for tok, t, p in zip(tokens, true_seq, pred_seq):
        marker = "" if t == p else "  <-- MISMATCH"
        print(f"    {tok:15s} true={t:10s} pred={p:10s}{marker}")
    print()
    shown += 1

Mismatched records: 14 / 272

id=453_aug1
    eh              true=O          pred=O         
    yang            true=O          pred=O         
    anu             true=O          pred=O         
    rekomendasinya  true=O          pred=O         
    pempek          true=B-FOOD_ITEM pred=B-FOOD_ITEM
    atau            true=O          pred=O         
    bakso           true=B-FOOD_ITEM pred=B-FOOD_ITEM
    beranak         true=I-FOOD_ITEM pred=I-FOOD_ITEM
    ya              true=O          pred=O         
    eh              true=O          pred=O         
    kurang          true=O          pred=O         
    satu            true=B-QUANTITY pred=B-QUANTITY
    ada             true=O          pred=O         
    yang            true=O          pred=O         
    unik-unik       true=O          pred=O         
    nggak           true=O          pred=O         
    kalo            true=O          pred=O         
    dihangat        true=B-MODIFIER pred=O           <-- MISMATCH

i

## Step 6 — Inference

Load the saved model fresh from disk to confirm it's reloadable standalone. `predict_entities(text)` runs raw text through tokenize -> predict -> decode-to-words, then extracts contiguous `B-X`/`I-X` spans per entity type — the structured-order output downstream LangGraph nodes will consume.

In [17]:
from transformers import AutoTokenizer as _AT

INFERENCE_MODEL_PATH = "../models/indobert-ner-bio-final"

inference_tokenizer = _AT.from_pretrained(INFERENCE_MODEL_PATH)
inference_model = BertCrfForTokenClassification.from_pretrained(INFERENCE_MODEL_PATH)
inference_model.to(device)
inference_model.eval()

print(f"Loaded model from {INFERENCE_MODEL_PATH}")
print(f"id2label: {inference_model.config.id2label}")


Loaded model from ../models/indobert-ner-bio-final
id2label: {0: 'O', 1: 'B-FOOD_ITEM', 2: 'I-FOOD_ITEM', 3: 'B-DRINK_ITEM', 4: 'I-DRINK_ITEM', 5: 'B-QUANTITY', 6: 'I-QUANTITY', 7: 'B-MODIFIER', 8: 'I-MODIFIER'}


In [18]:
def predict_entities(text: str) -> dict:
    """
    Run raw text through the fine-tuned entity tagger.

    Returns:
      tokens : word-level tokens (text.split())
      labels : predicted BIO tag per word (first-subword label only)
      spans  : list of {label: <TAG>, text: str, start: int, end: int}
               (end is exclusive, word indices)
    """
    tokens = text.lower().split()
    if not tokens:
        return {"tokens": [], "labels": [], "spans": []}

    encoding = inference_tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    word_ids = encoding.word_ids()
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        pred_ids = inference_model.decode(encoding["input_ids"], encoding["attention_mask"])[0]

    word_labels = [None] * len(tokens)
    prev_word_id = None
    for tok_idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id != prev_word_id:
            word_labels[word_id] = id2label[pred_ids[tok_idx]]
        prev_word_id = word_id
    word_labels = [lbl if lbl is not None else "O" for lbl in word_labels]

    spans = []
    i = 0
    while i < len(tokens):
        lbl = word_labels[i]
        if lbl.startswith("B-"):
            span_type = lbl[2:]
            j = i + 1
            while j < len(tokens) and word_labels[j] == f"I-{span_type}":
                j += 1
            spans.append({
                "label": span_type,
                "text": " ".join(tokens[i:j]),
                "start": i,
                "end": j,
            })
            i = j
        else:
            i += 1

    return {"tokens": tokens, "labels": word_labels, "spans": spans}

### Try it on fresh, unseen sentences

In [19]:
demo_sentences = [
    "saya mau pesan ayam bakar dua porsi pedas banget",
    "minta es teh dua gelas ya bang",
    "tolong nasi goreng satu porsi kecil aja",
    "eh saya mau ganti jadi mie ayam tiga porsi sama jus jeruk satu",
]

for text in demo_sentences:
    result = predict_entities(text)
    print(f"input  : {text}")
    print(f"tagged : {' '.join(f'{t}[{l}]' if l != 'O' else t for t, l in zip(result['tokens'], result['labels']))}")
    print(f"spans  : {result['spans']}")
    print()

input  : saya mau pesan ayam bakar dua porsi pedas banget
tagged : saya mau pesan ayam[B-FOOD_ITEM] bakar[I-FOOD_ITEM] dua[B-QUANTITY] porsi pedas[B-MODIFIER] banget
spans  : [{'label': 'FOOD_ITEM', 'text': 'ayam bakar', 'start': 3, 'end': 5}, {'label': 'QUANTITY', 'text': 'dua', 'start': 5, 'end': 6}, {'label': 'MODIFIER', 'text': 'pedas', 'start': 7, 'end': 8}]

input  : minta es teh dua gelas ya bang
tagged : minta es teh[I-DRINK_ITEM] dua[B-QUANTITY] gelas ya bang
spans  : [{'label': 'QUANTITY', 'text': 'dua', 'start': 3, 'end': 4}]

input  : tolong nasi goreng satu porsi kecil aja
tagged : tolong nasi[B-FOOD_ITEM] goreng[I-FOOD_ITEM] satu[B-QUANTITY] porsi kecil aja
spans  : [{'label': 'FOOD_ITEM', 'text': 'nasi goreng', 'start': 1, 'end': 3}, {'label': 'QUANTITY', 'text': 'satu', 'start': 3, 'end': 4}]

input  : eh saya mau ganti jadi mie ayam tiga porsi sama jus jeruk satu
tagged : eh saya mau ganti jadi mie[B-FOOD_ITEM] ayam[I-FOOD_ITEM] tiga[B-QUANTITY] porsi sama jus[B-DRINK_